In [ ]:
import time
from pynq import Overlay, MMIO

# -------------------------------
# Load bitstream
# -------------------------------
ol = Overlay("car.bit", ignore_version=True)

# -------------------------------
# AXI IIC MMIO
# -------------------------------
iic = MMIO(ol.ip_dict["axi_iic_0"]["phys_addr"], 0x10000)
CR, SR, TX = 0x100, 0x104, 0x108

def wait_tx_empty():
    while (iic.read(SR) & 0x80) == 0:
        pass

def iic_reset():
    iic.write(CR, 0x02)
    time.sleep(0.001)
    iic.write(CR, 0x01 | 0x04)
    while (iic.read(SR) & 0x02):
        pass

def iic_start_write(addr):
    wait_tx_empty()
    iic.write(TX, 0x100 | (addr << 1))

def iic_write_byte(b, stop=False):
    wait_tx_empty()
    if stop:
        b |= 0x200
    iic.write(TX, b)

# -------------------------------
# OLED
# -------------------------------
OLED_ADDR = 0x3C

def oled_cmd(c):
    iic_start_write(OLED_ADDR)
    iic_write_byte(0x00)
    iic_write_byte(c, stop=True)
    time.sleep(0.001)

def oled_data(data):
    iic_start_write(OLED_ADDR)
    iic_write_byte(0x40)
    for i, b in enumerate(data):
        iic_write_byte(b, stop=(i == len(data)-1))
    time.sleep(0.001)

def oled_init():
    iic_reset()
    time.sleep(0.1)
    cmds = [
        0xAE,0xD5,0x80,0xA8,0x3F,0xD3,0x00,0x40,
        0x8D,0x14,0x20,0x00,0xA1,0xC0,0xDA,0x12,
        0x81,0xCF,0xD9,0xF1,0xDB,0x40,0xA4,0xA6,0xAF
    ]
    for c in cmds:
        oled_cmd(c)

def oled_clear():
    oled_cmd(0x21); oled_cmd(0); oled_cmd(127)
    oled_cmd(0x22); oled_cmd(0); oled_cmd(7)
    oled_data([0x00]*1024)

# Minimal font for GPS
font = {**{str(i):[0]*8 for i in range(10)},
        '.':[0,0,0,0,0,24,24,0],
        '-':[0,0,0,126,0,0,0,0],
        ':':[0,24,24,0,24,24,0,0],
        'L':[64,64,64,64,64,64,126,0],
        'A':[24,36,66,126,66,66,66,0],
        'T':[126,8,8,8,8,8,8,0],
        'O':[60,66,66,66,66,66,60,0],
        'N':[66,98,82,74,70,66,66,0],
        ' ':[0]*8}

def rotate(g):
    r=[0]*8
    for y in range(8):
        for x in range(8):
            if g[y]&(1<<x):
                r[x]|=(1<<(7-y))
    return r

def oled_text(x,y,t):
    col=x
    for c in t:
        g=rotate(font.get(c,[0]*8))
        oled_cmd(0xB0+y)
        oled_cmd(col&0x0F)
        oled_cmd(0x10|(col>>4))
        oled_data(g)
        col+=9

oled_init()
oled_clear()


In [ ]:
gps_uart = ol.gps_uart

def nmea_to_decimal(coord, direction):
    if not coord or not direction:
        return None
    whole, frac = coord.split('.')
    deg = int(whole[:-2])
    mins = float(whole[-2:] + '.' + frac)
    val = deg + mins / 60
    if direction in 'SW':
        val *= -1
    return round(val, 6)

def read_nmea():
    s=""
    for _ in range(200):
        if gps_uart.read(0x08) & 1:
            s += chr(gps_uart.read(0x00) & 0xFF)
        else:
            break
    return s

def parse_gpgga(line):
    if not line.startswith('$GPGGA'):
        return None
    p = line.split(',')
    return nmea_to_decimal(p[2],p[3]), nmea_to_decimal(p[4],p[5])


In [ ]:
while True:
    raw = read_nmea()
    for line in raw.splitlines():
        if line.startswith('$GPGGA'):
            lat, lon = parse_gpgga(line)
            if lat and lon:
                oled_clear()
                oled_text(0,2,f"LAT:{lat}")
                oled_text(0,4,f"LON:{lon}")
    time.sleep(0.5)
